## Data Loading & Preprocessing*

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
from pathlib import Path
list(Path("/content/drive").glob("*"))          # shows 'MyDrive'
list(Path("/content/drive/MyDrive/cs231n_project/geo50k").glob("*"))

[PosixPath('/content/drive/MyDrive/cs231n_project/geo50k/compressed_dataset')]

In [5]:
!pip install -q --upgrade "fsspec>=2024.3.0" datasets

In [13]:
import random, pathlib, csv, datasets

SAMPLE_N = 2_000
IMG_ROOT = "/content/drive/MyDrive/cs231n_project/geo50k/compressed_dataset"
SAMP_CSV = "/content/gpt_sample/sample_paths.csv"

# 1️⃣ load dataset (decodes images to PIL)
ds = datasets.load_dataset("imagefolder", data_dir=IMG_ROOT, split="train")

# 2️⃣ take a deterministic random sample
sample = ds.shuffle(seed=42).select(range(SAMPLE_N))

# 3️⃣ write CSV with image path + label string
pathlib.Path(SAMP_CSV).parent.mkdir(parents=True, exist_ok=True)
with open(SAMP_CSV, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["image", "label"])
    for rec in sample:
        img_path = rec["image"].filename            # <─ path comes from PIL
        label    = ds.features["label"].int2str(rec["label"])
        w.writerow([img_path, label])

print(f"✅  Wrote {SAMP_CSV} with {len(sample)} rows")


Resolving data files:   0%|          | 0/49997 [00:00<?, ?it/s]

✅  Wrote /content/gpt_sample/sample_paths.csv with 2000 rows


## GPT Call

In [20]:
%%writefile gpt_geo_eval.py
import argparse, base64, json, os, time, csv, openai
from tqdm import tqdm

PROMPT = """You are a geolocation expert.
For the image the user sends, reply **only** in JSON like:
{
  "top1": "Country",
  "top5": ["Country1","Country2","Country3","Country4","Country5"]
}
Return your five most likely countries ranked from most to least likely."""

def img_to_data_uri(path):
    import mimetypes, base64
    mime = mimetypes.guess_type(path)[0] or "image/jpeg"
    b64  = base64.b64encode(open(path, "rb").read()).decode()
    return f"data:{mime};base64,{b64}"

def ask_gpt(img_path, model="gpt-4o-mini"):
    resp = openai.chat.completions.create(
        model=model,
        temperature=0,
        max_tokens=50,
        messages=[
            {"role": "system", "content": PROMPT},
            {"role": "user",
             "content": [{"type": "image_url",
                          "image_url": {"url": img_to_data_uri(img_path),
                                        "detail": "low"}}]},
        ],
        response_format={"type": "json_object"},
    )
    return json.loads(resp.choices[0].message.content)

def main(csv_path: str,
         print_every: int = 100,
         sleep_sec: float = 1.2,
         jsonl_path: str = "gpt_results.jsonl",
         model_name: str = "gpt-4o-mini"):

    # ensure we don't overwrite previous runs accidentally
    if os.path.exists(jsonl_path):
        raise FileExistsError(f"{jsonl_path} already exists; "
                              "rename or move it before running again.")

    with open(csv_path) as f:
        rows = list(csv.DictReader(f))

    tot = top1 = top5 = 0
    start = time.time()

    with open(jsonl_path, "w") as fout:
        for row in tqdm(rows, total=len(rows)):
            res = ask_gpt(row["image"], model=model_name)

            label = row["label"]
            top1  += int(res["top1"] == label)
            top5  += int(label in res["top5"])
            tot   += 1

            # stream result to disk
            fout.write(json.dumps({
                "image": row["image"],
                "ground_truth": label,
                "gpt_top1": res["top1"],
                "gpt_top5": res["top5"]
            }) + "\n")

            # interim log
            if tot % print_every == 0:
                elapsed = time.time() - start
                print(f"[{time.strftime('%H:%M:%S')}] "
                      f"{tot}/{len(rows)}  "
                      f"top-1={top1/tot:.3f}  top-5={top5/tot:.3f}  "
                      f"{elapsed/60:.1f} min elapsed")

            time.sleep(sleep_sec)   # stay under rate-limit

    print("\n════ FINAL RESULTS ════")
    print(f" Samples evaluated : {tot}")
    print(f" Top-1 accuracy    : {top1/tot:.4f}")
    print(f" Top-5 accuracy    : {top5/tot:.4f}")
    print(f" Raw responses saved to → {jsonl_path}")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", required=True, help="CSV with columns image,label")
    ap.add_argument("--print-every", type=int, default=100,
                    help="Status update interval")
    args = ap.parse_args()
    main(args.csv, args.print_every)


Overwriting gpt_geo_eval.py


In [ ]:
# install once
!pip install -q openai tqdm pandas pillow

# export your key
import os
os.environ["OPENAI_API_KEY"] = "xxx"

# run, printing interim metrics every 100 images
!python gpt_geo_eval.py \
        --csv /content/gpt_sample/sample_paths.csv \
        --print-every 100


  5% 99/2000 [05:17<1:37:50,  3.09s/it][23:56:13] 100/2000  top-1=0.730  top-5=0.880  5.3 min elapsed
 10% 199/2000 [10:36<1:49:01,  3.63s/it][00:01:33] 200/2000  top-1=0.705  top-5=0.875  10.6 min elapsed
 15% 299/2000 [16:17<1:35:16,  3.36s/it][00:07:14] 300/2000  top-1=0.730  top-5=0.880  16.3 min elapsed
 20% 399/2000 [21:52<1:31:40,  3.44s/it][00:12:50] 400/2000  top-1=0.733  top-5=0.880  21.9 min elapsed
 25% 499/2000 [27:27<1:18:01,  3.12s/it][00:18:24] 500/2000  top-1=0.724  top-5=0.878  27.5 min elapsed
 30% 599/2000 [32:57<1:23:05,  3.56s/it][00:23:54] 600/2000  top-1=0.717  top-5=0.878  33.0 min elapsed
 35% 699/2000 [39:28<1:20:59,  3.74s/it][00:30:25] 700/2000  top-1=0.726  top-5=0.879  39.5 min elapsed
 40% 799/2000 [45:00<1:06:48,  3.34s/it][00:35:57] 800/2000  top-1=0.726  top-5=0.873  45.1 min elapsed
 45% 899/2000 [50:30<53:26,  2.91s/it][00:41:26] 900/2000  top-1=0.722  top-5=0.869  50.5 min elapsed
 50% 999/2000 [56:02<53:31,  3.21s/it][00:46:59] 1000/2000  top-1=0.

## Results Evaluation

In [27]:
import os, pathlib, json, pandas as pd

JSONL_PATH = "/content/drive/MyDrive/cs231n_project/gpt_results.jsonl"   # adjust if you saved elsewhere

print("Exists :", os.path.isfile(JSONL_PATH))
if os.path.isfile(JSONL_PATH):
    print("Size   :", os.path.getsize(JSONL_PATH), "bytes")

Exists : True
Size   : 475250 bytes


In [28]:
rows = [json.loads(l) for l in open(JSONL_PATH) if l.strip()]
df = pd.DataFrame(rows)
df.head()

,image,ground_truth,gpt_top1,gpt_top5
0,/content/drive/MyDrive/cs231n_project/geo50k/c...,Finland,Finland,"[Finland, Sweden, Norway, Estonia, Russia]"
1,/content/drive/MyDrive/cs231n_project/geo50k/c...,Israel,Israel,"[Israel, Palestine, Jordan, Lebanon, Egypt]"
2,/content/drive/MyDrive/cs231n_project/geo50k/c...,Madagascar,Madagascar,"[Madagascar, Comoros, Mauritius, Seychelles, M..."
3,/content/drive/MyDrive/cs231n_project/geo50k/c...,United States,Colombia,"[Colombia, Venezuela, Ecuador, Peru, Panama]"
4,/content/drive/MyDrive/cs231n_project/geo50k/c...,Serbia,United States,"[United States, Canada, Australia, New Zealand..."


In [16]:
import pandas as pd
sample_csv = "/content/drive/MyDrive/cs231n_project/gpt_sample/sample_paths.csv"   # or your path
df = pd.read_csv(sample_csv)
print(df.shape)     # rows , 2
df.head(10)         # first 10 rows

(2000, 2)


,image,label
0,/content/drive/MyDrive/cs231n_project/geo50k/c...,Finland
1,/content/drive/MyDrive/cs231n_project/geo50k/c...,Israel
2,/content/drive/MyDrive/cs231n_project/geo50k/c...,Madagascar
3,/content/drive/MyDrive/cs231n_project/geo50k/c...,United States
4,/content/drive/MyDrive/cs231n_project/geo50k/c...,Serbia
5,/content/drive/MyDrive/cs231n_project/geo50k/c...,Mexico
6,/content/drive/MyDrive/cs231n_project/geo50k/c...,Canada
7,/content/drive/MyDrive/cs231n_project/geo50k/c...,Italy
8,/content/drive/MyDrive/cs231n_project/geo50k/c...,Brazil
9,/content/drive/MyDrive/cs231n_project/geo50k/c...,France


In [29]:
from pathlib import Path
import shutil, os

# ── paths ───────────────────────────────────────────────────────────────
DRIVE_DIR   = "/content/drive/MyDrive/cs231n_project"      # destination root
SRC_SAMPLE  = "/content/gpt_sample"                        # folder with CSV + (maybe) jsonl
SRC_SCRIPT  = "/content/gpt_geo_eval.py"                   # evaluation script
SRC_JSONL   = "/content/gpt_results.jsonl"                 # raw GPT outputs (if still here)

# 1️⃣ make sure the destination exists
Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)

# 2️⃣ move or copy files/folders
def move(src, dst_dir):
    src_path = Path(src)
    if not src_path.exists():
        print(f"⏭️  {src} not found, skipping")
        return
    dst = Path(dst_dir) / src_path.name
    if dst.exists():
        print(f"⏭️  {dst} already exists, skipping")
    else:
        shutil.move(str(src_path), str(dst))
        print(f"✅ moved {src_path.name} → {dst}")

move(SRC_SAMPLE, DRIVE_DIR)
move(SRC_SCRIPT, DRIVE_DIR)
move(SRC_JSONL, DRIVE_DIR)

print("\nContents of target directory:")
for p in Path(DRIVE_DIR).iterdir():
    print(" •", p.name)


✅ moved gpt_sample → /content/drive/MyDrive/cs231n_project/gpt_sample
✅ moved gpt_geo_eval.py → /content/drive/MyDrive/cs231n_project/gpt_geo_eval.py
✅ moved gpt_results.jsonl → /content/drive/MyDrive/cs231n_project/gpt_results.jsonl

Contents of target directory:
 • gpt_baseline.ipynb
 • geo50k
 • kaggle.json
 • archive.zip
 • baseline_resnet.ipynb
 • gpt_sample
 • gpt_geo_eval.py
 • gpt_results.jsonl
